In [17]:
import glob
import json
import os
import sys
import time
from collections import deque
from pathlib import Path

import cudf
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import seaborn as sns
import shap
import wandb
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.append(os.path.abspath(".."))

from src.utils.print_duration import print_duration

In [29]:
# Categorical Dataの情報を取得

path = "sample.parquet"
pf = pq.ParquetFile(path)
schema = pf.schema_arrow

cats = []

for field in schema:
    is_cat_meta = False
    if field.metadata:  # dict[bytes, bytes]
        for k, v in field.metadata.items():
            kb = k or b""
            if b"CATEGORICAL" in kb:
                is_cat_meta = True
                break

    is_dict = pa.types.is_dictionary(field.type)

    if is_cat_meta or is_dict:
        cats.append(field.name)

print("CATS:", cats)

CATS: ['z']


In [ ]:
class CFG:
    COMPETITION = "binary-bank"
    DEBUG = False
    TUNING = False
    MODEL = "xgb"
    DATA_ID = "026"
    SEED = 42

In [ ]:
try:
    import cudf  # GPU 環境なら使う（なければ自動で pandas にフォールバック）

    _HAS_CUDF = True
except Exception:
    _HAS_CUDF = False


class ParquetIter(xgb.core.DataIter):
    """
    ディスク上の Parquet をバッチで読み出し、XGBoost に逐次供給するための DataIter。
    - path は str でも list[str] でもOK
    - features: 学習に使う特徴量カラム名リスト
    - target: 目的変数カラム名
    - batch_rows: 1バッチの行数上限（大きすぎるとGPU/CPUメモリに乗らないことがある）
    - use_cudf: True なら cuDF データフレームで渡す（GPU学習時に有利）
    """

    def __init__(
        self, path, features, target, cat_cols=None, batch_rows=1_000_000, use_cudf=None
    ):
        super().__init__()
        if isinstance(path, (str, os.PathLike)):
            path = [str(path)]
        self.paths = list(path)
        self.features = features
        self.cat_cols = cat_cols if cat_cols else []
        self.target = target
        self.batch_rows = int(batch_rows)
        self.use_cudf = _HAS_CUDF if use_cudf is None else bool(use_cudf)

        # 内部状態
        self._reader = None  # deque of RecordBatch
        self._current_file_index = 0  # どのファイルを読んでいるか

    # --- 必須: 反復の最初に呼ばれる ---
    def reset(self):
        self._current_file_index = 0
        self._reader = None

    # --- 必須: 次のバッチを input_data に詰めて 1 を返す。終端で 0 を返す ---
    def next(self, input_data):
        while True:
            if self._reader is None:
                if not self._prepare_next_file():
                    return 0
            try:
                batch = next(self._reader)
            except StopIteration:
                self._reader = None
                continue  # 次ファイルへ
            # Arrow -> pandas/cuDF
            df = (
                cudf.DataFrame.from_arrow(batch) if self.use_cudf else batch.to_pandas()
            )
            # カテゴリを数値化
            if self.cat_cols:
                df[self.cat_cols] = df[self.cat_cols].astype("category")
            input_data(data=df[self.features], label=df[self.target])
            del df  # 参照を切る（オプション）
            return 1

    # --- 内部: 次のファイルのバッチ列を準備する。準備できれば True ---
    def _prepare_next_file(self):
        while self._current_file_index < len(self.paths):
            path = self.paths[self._current_file_index]
            self._current_file_index += 1

            # 単一ファイルの dataset を作る（列を絞る）
            dataset = ds.dataset(path, format="parquet")
            cols = list(dict.fromkeys(self.features + [self.target]))

            self._reader = dataset.scanner(
                batch_size=self.batch_rows, columns=cols
            ).to_reader()
            return True
        self._reader = None
        return False

In [32]:
def make_fold_paths(base_dir, ID, SEED, valid_fold_idx=0):
    base_dir = Path(base_dir)
    pattern = str(base_dir / f"tr_df{ID}-fold*-seed{SEED}.parquet")
    all_files = sorted(glob.glob(pattern))
    print(f"We found {len(all_files)} files: {all_files}")

    # TrainとValidの仕分け
    train_files, valid_file = [], None
    train_idx = []
    valid_idx = []
    for p in all_files:
        name = Path(p).name
        try:
            f = int(name.split("-fold")[1].split("-")[0])
        except Exception:
            continue
        if f == valid_fold_idx:
            valid_file = p
            valid_idx.append(f)
        else:
            train_files.append(p)
            train_idx.append(f)

    if valid_file is None:
        raise FileNotFoundError(f"valid fold file not found for fold={valid_fold_idx}")

    print(f"Train files index: {train_idx}")
    print(f"Valid files index: {valid_idx}")

    return train_files, valid_file

In [34]:
import numpy as np
import pyarrow.dataset as ds

def collect_row_ids_for_fold(paths, fold_col, fold_idx, batch_rows=1_000_000):
    if isinstance(paths, (str, os.PathLike)):
        paths = [str(paths)]
    dataset = ds.dataset(paths, format="parquet")
    reader = dataset.scanner(
        columns=["row_id"],
        filter=(ds.field(fold_col) == fold_idx),
        batch_size=batch_rows,
    ).to_reader()

    chunks = []
    for batch in reader:
        # Arrow Array -> numpy
        arr = batch.column(0)  # row_id
        chunks.append(np.asarray(arr))
    return np.concatenate(chunks) if chunks else np.empty(0, dtype=np.int64)

We found 5 files: ['../artifacts/features/base/026/tr_df026-fold0-seed42.parquet', '../artifacts/features/base/026/tr_df026-fold1-seed42.parquet', '../artifacts/features/base/026/tr_df026-fold2-seed42.parquet', '../artifacts/features/base/026/tr_df026-fold3-seed42.parquet', '../artifacts/features/base/026/tr_df026-fold4-seed42.parquet']
Train files index: [1, 2, 3, 4]
Valid files index: [0]


In [ ]:
def class2dict(f):
    return dict(
        (name, getattr(f, name)) for name in dir(f) if not name.startswith("__")
    )

In [ ]:
class XGBCVTrainer:
    """
    XGBを使ったCVトレーナー。

    Attributes
    ----------
    tr_df : pd.DataFrame
        label付データ
    test_df : pd.DataFrame, default None
        labelなしデータ。CV学習とFull Trainはtest_df必須。
    params : dict
        XGBのパラメータ。
    n_splits : int, default 5
        StratifiedKFoldの分割数。
    early_stopping_rounds : int, default 100
        早期停止ラウンド数。
    num_boost_round : int, default 20000
        iterationの最大値。
    seed : int, default 42
        乱数シード。
    """

    def __init__(
        self,
        tr_df,
        test_df=None,
        params=None,
        n_splits=5,
        early_stopping_rounds=200,
        num_boost_round=20000,
        seed=42,
    ):
        self.params = params
        self.n_splits = n_splits
        self.early_stopping_rounds = early_stopping_rounds
        self.num_boost_round = num_boost_round
        self.fold_models = []
        self.fold_scores = []
        self.seed = seed
        self.oof_score = None

        # object → category
        cat_cols = tr_df.select_dtypes(include="object").columns
        tr_df[cat_cols] = tr_df[cat_cols].astype("category")

        # target
        self.X = tr_df.drop("target", axis=1)
        self.y = tr_df["target"].to_numpy()

        # test
        if test_df is not None:
            test_df[cat_cols] = test_df[cat_cols].astype("category")
            self.test = xgb.DMatrix(test_df, enable_categorical=True)
        else:
            self.test = None

        # fold indices
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=self.seed)
        self.fold_indices = list(skf.split(self.X, self.y))

        self.default_params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "learning_rate": 0.1,
            "max_depth": 7,
            "min_child_weight": 10.0,
            "gamma": 0,
            "colsample_bytree": 0.8,
            "subsample": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "verbosity": 0,
            "tree_method": "hist",
            "device": "cuda",
            "random_state": self.seed,
            "max_bin": 512,
            "grow_policy": "depthwise",
            "single_precision_histogram": True,
            "predictor": "gpu_predictor",
        }
        self.params = {**self.default_params, **(self.params or {})}

    def fit(self):
        """
        CVを用いてモデルを学習し、OOF予測とtest_dfの平均予測を返す。

        Returns
        -------
        oof_preds : ndarray
            OOF予測配列
        test_preds : ndarray
            test_dfに対する予測配列
        """
        if self.test is None:
            raise ValueError("test_df not provided for XGBCVTrainer.")

        oof_preds = np.zeros(len(self.X))
        test_preds = np.zeros(self.test.num_row())

        iteration_list = []

        for fold, (tr_idx, val_idx) in enumerate(self.fold_indices):
            print(f"\nFold {fold + 1}")
            start = time.time()

            X_tr, y_tr, w_tr = (
                self.X.iloc[tr_idx],
                self.y[tr_idx],
                self.weights[tr_idx],
            )
            X_val, y_val = self.X.iloc[val_idx], self.y[val_idx]

            dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, enable_categorical=True)
            dvalid = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
            evals_result = {}

            model = xgb.train(
                self.params,
                dtrain,
                num_boost_round=self.num_boost_round,
                evals=[(dtrain, "train"), (dvalid, "eval")],
                early_stopping_rounds=self.early_stopping_rounds,
                verbose_eval=100,
                evals_result=evals_result,
            )

            # oof
            oof_preds[val_idx] = model.predict(
                dvalid, iteration_range=(0, model.best_iteration + 1)
            )
            test_preds += model.predict(
                self.test, iteration_range=(0, model.best_iteration + 1)
            )

            end = time.time()
            print_duration(start, end)

            best_iter = model.best_iteration
            train_score = evals_result["train"]["auc"][best_iter]
            eval_score = evals_result["eval"]["auc"][best_iter]
            print(f"Train AUC: {train_score:.5f}")
            print(f"Valid AUC: {eval_score:.5f}")

            self.fold_scores.append(eval_score)

            iteration_list.append(best_iter)

        print("\n=== CV Results ===")
        print(f"Fold scores: {self.fold_scores}")
        print(
            f"Mean: {np.mean(self.fold_scores):.5f}, "
            f"Std: {np.std(self.fold_scores):.5f}"
        )

        self.oof_score = roc_auc_score(self.y, oof_preds)
        print(f"OOF score: {self.oof_score:.5f}")
        print(f"Avg best iteration: {np.mean(iteration_list)}")
        print(f"Best iterations: \n{iteration_list}")

        test_preds /= self.n_splits

        return oof_preds, test_preds

    def fit_one_fold(self, fold=0):
        """
        指定した1つのfoldのみを用いてモデルを学習する。
        主にOptunaによるハイパーパラメータ探索時に使用。

        Parameters
        ----------
        fold : int
            学習に使うfold番号。

        Rerurn
        ------
        score : float
            Score
        """
        tr_idx, val_idx = self.fold_indices[fold]
        start = time.time()

        X_tr, y_tr, w_tr = self.X.iloc[tr_idx], self.y[tr_idx], self.weights[tr_idx]
        X_val, y_val = self.X.iloc[val_idx], self.y[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, enable_categorical=True)
        dvalid = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

        evals_result = {}

        model = xgb.train(
            self.params,
            dtrain,
            num_boost_round=self.num_boost_round,
            evals=[(dtrain, "train"), (dvalid, "eval")],
            early_stopping_rounds=self.early_stopping_rounds,
            verbose_eval=100,
            evals_result=evals_result,
            callbacks=[wandb.xgboost.WandbCallback()],
        )

        end = time.time()
        print_duration(start, end)

        best_iter = model.best_iteration
        train_score = evals_result["train"]["auc"][best_iter]
        eval_score = evals_result["eval"]["auc"][best_iter]
        print(f"Train AUC: {train_score:.5f}")
        print(f"Valid AUC: {eval_score:.5f}")

        return eval_score